# Encore: rotation and survival

Two questions about a band's live repertoire.

1. **Do the shows change from night to night?** *Rotation* compares each show
   with the next one of the same tour: `1 - Jaccard` of the two sets of songs
   (0 = identical setlists, 1 = nothing in common).
2. **How long does a song survive in the setlist?** Duration is counted in the
   band's shows from a song's live debut. A song is *abandoned* when it leaves
   the setlist and does not return: its last appearance is at least **N**
   shows (N = 50 here) before the end of the history. A song still being
   played, or one that was away for a while and came back, is *censored*.
   Curves are Kaplan-Meier estimates.

This notebook reads **only the aggregated marts in the `analytics` schema**
(`mart_band_rotation_by_year`, `mart_survival_curves`, `mart_survival_summary`).
It never touches raw data, and it holds no setlists, dates, venues or song
titles. Definitions and limitations: `docs/methodology.md`.

Run the pipeline first so the marts are populated. To preview against another
Postgres instance, set `ENCORE_DB_HOST` / `ENCORE_DB_PORT` (see Setup).

Show data: [setlist.fm](https://www.setlist.fm) (aggregated only). Discography: [MusicBrainz](https://musicbrainz.org) (CC0).

## Setup

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import psycopg2
from dotenv import load_dotenv

# Repo root = first parent folder holding CLAUDE.md (works from notebooks/ or the root).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())
load_dotenv(ROOT / ".env")

# Postgres is published on the host at 127.0.0.1:5435 (docs/context/tech.md).
# ENCORE_DB_HOST / ENCORE_DB_PORT let a dev preview point at another instance.
DB = dict(
    host=os.environ.get("ENCORE_DB_HOST", "localhost"),
    port=int(os.environ.get("ENCORE_DB_PORT", "5435")),
    user=os.environ["POSTGRES_USER"],
    password=os.environ["POSTGRES_PASSWORD"],
    dbname="encore",
)

# The only tables this notebook may read. Anything else is refused.
ANALYTICS_TABLES = {
    "mart_band_rotation_by_year", "mart_survival_curves", "mart_survival_summary",
}
TEXT_COLUMNS = {"band", "album"}


def read_analytics(table: str) -> pd.DataFrame:
    """Read one analytics mart through a read-only session."""
    if table not in ANALYTICS_TABLES:
        raise ValueError(f"{table!r} is not an allowed analytics table")
    conn = psycopg2.connect(**DB)
    try:
        conn.set_session(readonly=True)
        with conn.cursor() as cur:
            cur.execute(f"select * from analytics.{table}")
            columns = [c.name for c in cur.description]
            df = pd.DataFrame(cur.fetchall(), columns=columns)
    finally:
        conn.close()
    df = df.drop(columns=["computed_at"])
    # Postgres numeric arrives as Decimal; make the numbers plain floats/ints.
    return df.apply(lambda s: s if s.name in TEXT_COLUMNS else pd.to_numeric(s))


# One fixed colour per band (the colour follows the band, never its rank), in
# the project's band order. Light-surface values of the reference categorical palette.
BANDS = [
    "Arctic Monkeys", "Oasis", "Linkin Park", "Twenty One Pilots",
    "Muse", "Metallica", "Avenged Sevenfold",
]
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
BAND_COLOR = dict(zip(BANDS, PALETTE))
INK, INK_MUTED, GRID, NEUTRAL, SURFACE = "#0b0b0b", "#52514e", "#e6e5e1", "#a3a29c", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_MUTED,
    "xtick.color": INK_MUTED, "ytick.color": INK_MUTED, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "font.size": 10,
})

rotation = read_analytics("mart_band_rotation_by_year")
curves = read_analytics("mart_survival_curves")
summary = read_analytics("mart_survival_summary")
print(f"mart_band_rotation_by_year: {len(rotation)} rows | "
      f"mart_survival_curves: {len(curves)} rows | mart_survival_summary: {len(summary)} rows")
print("bands present:", ", ".join(sorted(rotation["band"].unique())))

## 1. Rotation by year, one panel per band

Each point is the average rotation of a band's consecutive show pairs in that
year (a pair belongs to the year of its later show), so a higher value means
setlists change more from one night to the next. All panels share the same
vertical scale (0 to 1), so the bands can be compared by eye. A year with
**fewer than 5 pairs** is too thin to trust: it is drawn as a hollow marker
and left out of the connecting line. Tours with fewer than 5 shows are not
scored at all.

In [ ]:
MIN_PAIRS = 5  # years with fewer show pairs are drawn hollow, outside the line

bands_present = [b for b in BANDS if b in set(rotation["band"])]
ncols = 4
nrows = -(-len(bands_present) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 2.9 * nrows), sharey=True, squeeze=False)
for ax, band in zip(axes.flat, bands_present):
    color = BAND_COLOR[band]
    data = rotation[rotation["band"] == band].set_index("show_year").sort_index()
    years = range(int(data.index.min()), int(data.index.max()) + 1)
    solid = data["rotation"].where(data["pairs"] >= MIN_PAIRS).reindex(years)
    ax.plot(solid.index, solid.values, color=color, linewidth=2, marker="o", markersize=6,
            markeredgecolor=SURFACE, markeredgewidth=1.2)
    thin = data[data["pairs"] < MIN_PAIRS]
    ax.plot(thin.index, thin["rotation"], linestyle="none", marker="o", markersize=6,
            markerfacecolor=SURFACE, markeredgecolor=color, markeredgewidth=1.6)
    ax.set_title(band, loc="left", fontsize=11, color=INK)
    ax.set_ylim(-0.03, 1.03)
    ax.set_xlim(years[0] - 0.5, years[-1] + 0.5)
    ax.xaxis.set_major_locator(plt.MaxNLocator(nbins=4, integer=True))
    ax.ticklabel_format(useOffset=False, axis="x")
    if len(years) == 1:
        ax.set_xticks(list(years))
for ax in list(axes.flat)[len(bands_present):]:
    ax.set_visible(False)
for ax in axes[:, 0]:
    ax.set_ylabel("Rotation (0 = same setlist)")
fig.suptitle("Rotation between consecutive shows, by year", x=0.01, ha="left", fontsize=12, color=INK)
fig.text(0.01, 0.955, f"Years with fewer than {MIN_PAIRS} show pairs: hollow markers, left out of the line.",
         fontsize=9, color=INK_MUTED, va="top")
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

## 2. How long do songs last? Kaplan-Meier curves, one band

Set `BAND` to any band present in the data. Each line is one album: the share
of that album's songs still in the live repertoire after *t* shows since their
live debut, with a shaded 95% confidence band. `non-album` is the songs matched
to the catalog but not to a studio album; the dashed line is every eligible song
of the band together. Only albums with **at least 5 songs** are drawn: with
fewer, the confidence band says nothing. The curve steps down each time songs
are abandoned (last appearance 50 or more shows before the end of the history)
and never drops for songs still being played or that came back (censored).

In [ ]:
BAND = "Oasis"
N_WINDOW = 50
MIN_SONGS = 5  # albums with fewer songs are not drawn

sizes = summary[(summary["band"] == BAND) & (summary["n_window"] == N_WINDOW)].set_index("album")["songs"]
albums = sorted(a for a in sizes.index if a not in ("all", "non-album") and sizes[a] >= MIN_SONGS)
if "non-album" in sizes.index and sizes["non-album"] >= MIN_SONGS:
    albums.append("non-album")
# Album colours follow the (alphabetical) album, in the fixed palette order; non-album is grey.
album_color = {a: (NEUTRAL if a == "non-album" else PALETTE[i % len(PALETTE)]) for i, a in enumerate(albums)}


def curve(album: str) -> pd.DataFrame:
    c = curves[(curves["band"] == BAND) & (curves["n_window"] == N_WINDOW) & (curves["album"] == album)]
    return c.sort_values("t_shows")


fig, ax = plt.subplots(figsize=(11.5, 5))
for album in albums:
    c = curve(album)
    color = album_color[album]
    ax.fill_between(c["t_shows"], c["ci_lower"], c["ci_upper"], step="post", color=color, alpha=0.12, linewidth=0)
    ax.step(c["t_shows"], c["survival_probability"], where="post", color=color, linewidth=2,
            label=f"{album} ({int(sizes[album])} songs)")
overall = curve("all")
if len(overall):
    ax.step(overall["t_shows"], overall["survival_probability"], where="post", color=INK,
            linewidth=1.6, linestyle="--", label=f"All eligible songs ({int(sizes['all'])})")
ax.axhline(0.5, color=NEUTRAL, linewidth=0.8)
ax.text(1.0, 0.5, "median ", transform=ax.get_yaxis_transform(), ha="right", va="bottom", fontsize=9, color=INK_MUTED)
ax.set_title(f"{BAND}: share of songs still in the setlist after t shows", loc="left", fontsize=12, color=INK, pad=24)
ax.text(0, 1.025, f"Abandoned = left and did not return (last played {N_WINDOW}+ shows before the end). Shaded: 95% confidence band. Albums with fewer than {MIN_SONGS} songs are not drawn.",
        transform=ax.transAxes, fontsize=9, color=INK_MUTED, va="bottom")
ax.set_xlabel("Shows since the song's live debut")
ax.set_ylabel("Share of songs still played")
ax.set_ylim(0, 1.02)
ax.set_xlim(left=0)
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
fig.tight_layout()
plt.show()

## 3. Median survival by band and album (N = 50)

The median is the number of shows after which half of the songs have been
abandoned. **Not reached** means fewer than half of the songs were abandoned
by the end of the history, so the curve never crosses 0.5 (the songs mostly
survive, or are too recent to tell). `songs` = eligible songs (at least 3
performances, matched to an album or to a dated recording); `abandoned` are the
ones that left the setlist; `censored` are still being played. `all` is every
eligible song of the band. Albums with fewer than 5 songs are marked with an
asterisk: read them with care.

In [ ]:
table = summary[summary["n_window"] == N_WINDOW].copy()
table["band"] = pd.Categorical(table["band"], categories=BANDS, ordered=True)
table["is_album"] = table["album"] != "all"  # the band total first, then albums
table = table.sort_values(["band", "is_album", "album"]).drop(columns=["is_album", "n_window"])
table["album"] = table.apply(
    lambda r: r["album"] + (" *" if r["album"] != "all" and r["songs"] < MIN_SONGS else ""), axis=1)
table["median (shows)"] = table["median_survival_shows"].map(
    lambda v: "not reached" if pd.isna(v) else f"{v:.0f}")
display(
    table.rename(columns={"events": "abandoned"})[["band", "album", "songs", "abandoned", "censored", "median (shows)"]]
    .style.format({"songs": "{:,}", "abandoned": "{:,}", "censored": "{:,}"})
    .hide(axis="index")
)